# Notebook 01 — Data Loading, Preprocessing & Exploration

**Dataset:** 10x Genomics Xenium In Situ — Human Breast Cancer (FFPE)  
**Goal:** Load all relevant data files, perform quality filtering, and visualise the spatial distribution of transcripts across the tissue section.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import tifffile
import squidpy as sq
import scanpy as sc
import anndata as ad
from pathlib import Path

from src.visualization.spatial_plots import plot_transcript_density

sc.settings.verbosity = 1
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Load Transcript Coordinates

In [ ]:
transcripts = pd.read_csv(DATA_DIR / "transcripts.csv.gz", compression="gzip")
print(f"Total transcripts: {len(transcripts):,}")
print(f"Columns: {list(transcripts.columns)}")
transcripts.head()

## 2. Quality Filtering

Xenium assigns a Q-score to each decoded transcript. Q ≥ 20 is the standard threshold used by 10x.

In [ ]:
print(f"Q-score distribution:\n{transcripts['qv'].describe()}\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(transcripts["qv"], bins=50, color="steelblue", edgecolor="none")
axes[0].axvline(20, color="red", linestyle="--", label="Q20 threshold")
axes[0].set_xlabel("Q-score")
axes[0].set_ylabel("Count")
axes[0].set_title("Q-score Distribution")
axes[0].legend()

# Before / after filtering
transcripts_filtered = transcripts[transcripts["qv"] >= 20].copy()
labels = ["Raw", "Q ≥ 20"]
counts = [len(transcripts), len(transcripts_filtered)]
axes[1].bar(labels, counts, color=["steelblue", "seagreen"])
axes[1].set_ylabel("Transcript count")
axes[1].set_title("Before vs. After Quality Filtering")
for i, v in enumerate(counts):
    axes[1].text(i, v + 5000, f"{v:,}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "qscore_filtering.png", dpi=150)
plt.show()

pct_retained = 100 * len(transcripts_filtered) / len(transcripts)
print(f"Retained {len(transcripts_filtered):,} transcripts ({pct_retained:.1f}%) after Q ≥ 20 filter.")

## 3. Gene Panel Overview

In [ ]:
gene_counts = transcripts_filtered["feature_name"].value_counts()
print(f"Genes detected: {len(gene_counts)}")
print(f"\nTop 10 most detected genes:")
print(gene_counts.head(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
gene_counts.head(30).plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_xlabel("Gene")
ax.set_ylabel("Transcript count")
ax.set_title("Top 30 Most Detected Genes")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "top_genes.png", dpi=150)
plt.show()

## 4. Spatial Distribution of Transcripts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
plot_transcript_density(
    transcripts_filtered,
    x_col="x_location",
    y_col="y_location",
    ax=ax,
    title="Spatial Distribution of All Transcripts (Q ≥ 20)",
    s=0.2,
    alpha=0.15,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "transcript_spatial_distribution.png", dpi=150)
plt.show()

## 5. Marker Gene Visualisation

Visualise key cancer tissue markers to confirm expected spatial patterns.

In [ ]:
marker_genes = ["EPCAM", "CD68", "CD3E", "KRT17"]
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

for ax, gene in zip(axes, marker_genes):
    subset = transcripts_filtered[transcripts_filtered["feature_name"] == gene]
    ax.scatter(
        subset["x_location"],
        subset["y_location"],
        s=0.3,
        alpha=0.4,
        color="tomato",
    )
    ax.set_title(f"{gene} ({len(subset):,} transcripts)")
    ax.set_aspect("equal")
    ax.axis("off")

fig.suptitle("Marker Gene Spatial Patterns", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "marker_genes_spatial.png", dpi=150)
plt.show()

## 6. Load 10x Nucleus Boundaries (Reference / Ground Truth)

In [ ]:
nucleus_boundaries = pd.read_csv(DATA_DIR / "nucleus_boundaries.csv.gz", compression="gzip")
print(f"Nucleus boundary records: {len(nucleus_boundaries):,}")
print(f"Unique nuclei: {nucleus_boundaries['cell_id'].nunique():,}")
nucleus_boundaries.head()

## 7. Load 10x AnnData (Cell × Gene Matrix)

In [ ]:
adata = sc.read_10x_h5(DATA_DIR / "cell_feature_matrix.h5")
cells_df = pd.read_csv(DATA_DIR / "cells.csv.gz", compression="gzip", index_col=0)
adata.obs = adata.obs.join(cells_df, how="left")
adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].values
print(adata)
adata

## 8. Save Filtered Transcripts

In [ ]:
transcripts_filtered.to_parquet(DATA_DIR / "transcripts_filtered.parquet", index=False)
adata.write_h5ad(DATA_DIR / "xenium_adata.h5ad")
print("Saved filtered transcripts and AnnData.")